# Customer Segmentation — RFM Analysis

Only **3.0%** of customers buy more than once (see `01_eda.ipynb`), so knowing *which* customers are valuable — and which valuable ones are slipping away — is the highest-leverage question for this business.

**RFM** scores every customer on:
- **R**ecency — days since last purchase (lower is better)
- **F**requency — number of orders
- **M**onetary — total spend (R$)

Each dimension is split into quintiles (1–5), then mapped to business segments. A KMeans check at the end validates that the segments reflect real structure.

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import plotly.express as px
from ecommerce import build_rfm

rfm = build_rfm("../data")
print(rfm.shape)
print(rfm[["recency_days", "frequency", "monetary"]].describe().round(1))

## RFM distributions

In [ ]:
for col, title in [("recency_days", "Recency (days since last purchase)"),
                   ("frequency", "Frequency (orders per customer)"),
                   ("monetary", "Monetary (total spend, R$")]:
    fig = px.histogram(rfm, x=col, nbins=50, title=title)
    fig.show()

## Business segments

In [ ]:
seg = (rfm.groupby("segment")
        .agg(customers=("customer_unique_id", "count"),
             revenue=("monetary", "sum"),
             avg_recency=("recency_days", "mean"),
             avg_frequency=("frequency", "mean"),
             avg_monetary=("monetary", "mean"))
        .sort_values("customers", ascending=False))
seg["cust_share"] = seg["customers"] / seg["customers"].sum()
seg["rev_share"] = seg["revenue"] / seg["revenue"].sum()
print(seg.round(2))
fig = px.bar(seg.reset_index(), x="segment", y="customers", color="rev_share",
             color_continuous_scale="Blues",
             title="RFM segments — customer count coloured by revenue share",
             labels={"segment": "", "customers": "Customers"})
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## Sanity check: does KMeans find similar structure?

KMeans (k=5) on log-scaled, standardised RFM — a completely different method. If the clusters line up with the RFM segments, the segmentation is robust.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = StandardScaler().fit_transform(
    np.log1p(rfm[["recency_days", "frequency", "monetary"]]))
rfm["cluster"] = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X)
prof = (rfm.groupby("cluster")
           .agg(customers=("customer_unique_id", "count"),
                avg_recency=("recency_days", "mean"),
                avg_frequency=("frequency", "mean"),
                avg_monetary=("monetary", "mean"))
           .round(1).sort_values("avg_monetary", ascending=False))
print(prof)
print("\nCluster vs RFM segment crosstab:")
print(pd.crosstab(rfm["cluster"], rfm["segment"]))

## What to do with each segment

- **At Risk (22,357 customers · 24% of revenue, R$ 3.7M)** — the single biggest lever. They bought often and spent well but went quiet. Win-back campaign: personalised offers, reminders of categories they bought.
- **Champions (14,871 · R$ 2.6M)** — recent, frequent, high spend. VIP treatment, early access, referral incentives; they are the cheapest source of new customers.
- **Loyal Customers (14,993 · R$ 2.5M)** — buy regularly but slightly less recently. Upsell/cross-sell into adjacent categories.
- **New Customers (7,479)** — bought once, recently. With a 3% repeat rate, converting even a fraction to a second purchase is transformative: post-purchase onboarding + timed discount.
- **Hibernating (7,503) + Lost (7,483)** — long gone. Cheap reactivation test on a sample; don't overspend here.
- **Need Attention (7,373)** — average on everything and drifting. Nurture before they slide into At Risk.

Bottom line: retention marketing aimed at **At Risk + New Customers** protects ~R$ 5M of revenue and attacks the business's core weakness — the 3% repeat-purchase rate.